# 2 · Benthic habitat classification of Tongatapu

The Python counterpart of GEE **script 2**, with stronger training and testing.

| Step | GEE script 2 | Here |
|---|---|---|
| Training data | 300 random points per class inside the polygons | drop-camera-style points: one class per drop, with GPS error |
| Exploring the data | — | spectra by class, deep water mask check, spatial autocorrelation |
| Features | B1–B5, 6 Lyzenga DIIs, 3 ratio indices | same |
| Classifier | Random Forest (50 trees), seagrass, repeated 20 times | Random Forest (50 trees), all six classes, three ways: pixels, superpixels (two-stage), foundation model embeddings |
| Testing | random 70 / 30 split | held-out **spatial blocks**, plus random vs spatial cross-validation |
| Where to trust the map | — | **area of applicability** |

**Inputs:** `data/tongatapu_composite.nc` (notebook 1), `tongatapu_drops.geojson`
**Outputs:** `data/tongatapu_classes_{pixel,object,embedding}.tif`, `data/rf_models.joblib`

The embeddings (section 8c) are read from the internet, about 0.5 GB.

In [ ]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # noqa: F401
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from shapely.geometry import MultiPolygon, Polygon
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold, PredefinedSplit, cross_val_predict
from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score, f1_score, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance
from skimage.color import rgb2hsv
from skimage.segmentation import slic, find_boundaries
from shapely.geometry import box
from rasterio.enums import Resampling
import urllib.request
import joblib

CRS = "EPSG:32701"
CLASSES = ["Algal mat/sparse seagrass", "Seagrass", "Sand", "Rubble", "Coral", "Deep Water"]
COLOURS = ["#b5e48c", "#2d6a4f", "#e9c46a", "#e9a07a", "#c9577a", "#1f4e9c"]
BANDS = {"B1": 443, "B2": 490, "B3": 560, "B4": 665, "B5": 705, "B8A": 865, "B11": 1610}   # band: wavelength (nm)
N_TREES = 50
MIN_DROPS = 15        # classes with fewer drops (after deep water masking) are left out

DATA = Path("data")
with xr.open_dataset(DATA / "tongatapu_composite.nc", engine="h5netcdf") as f:      # read, then close the file
    comp = f.load()
comp = comp.astype("float32").rio.write_crs(CRS)
drops = gpd.read_file("tongatapu_drops.geojson").to_crs(CRS)

def rgb(img, gain):
    return np.clip(np.nan_to_num(np.dstack([img["B4"], img["B3"], img["B2"]]) * gain), 0, 1)

## 1. The training data: drop-camera observations

Each record is one camera drop: an ID, the survey day and time, the **majority class** seen on the seabed, and the boat's GPS position. Drops were made over 23 survey days between July and September 2023, and are at least ~40 m apart. Some classes have far fewer drops than others, as in most real surveys.

> These points were **generated for this exercise** from expert-drawn habitat polygons (`prep/make_drop_points.py`), to resemble a real drop-camera survey: one label per drop, positions from a boat GPS with a few metres of error. They are not real observations.

In [ ]:
x0, y0, x1, y1 = drops.total_bounds + [-1500, -1500, 1500, 1500]
view = comp.sel(x=slice(x0, x1), y=slice(y1, y0))
half = abs(float(comp.x[1] - comp.x[0])) / 2
x0, x1, y0, y1 = float(view.x[0]) - half, float(view.x[-1]) + half, float(view.y[-1]) - half, float(view.y[0]) + half   # extent of the pixels actually returned
fig, ax = plt.subplots(figsize=(15, 7))
ax.imshow(rgb(view, 4), extent=[x0, x1, y0, y1])
for cls, colour in zip(CLASSES, COLOURS):
    d = drops[drops["class"] == cls]
    ax.scatter(d.geometry.x, d.geometry.y, s=14, color=colour, edgecolor="k", lw=0.3, label=f"{cls} ({len(d)})")
ax.legend(fontsize=8, loc="lower left"); ax.set_title(f"{len(drops)} camera drops"); ax.axis("off")

drops.groupby("class").agg(drops=("drop_id", "size"), survey_days=("survey_day", "nunique")).reindex(CLASSES)

## 2. Check the deep water mask

The drops are an independent check on notebook 1's `DEEP_THRESHOLD`. Nearly all drops of the shallow classes should lie **outside** the deep water mask. Some `Deep Water` drops will lie inside it, which is expected. The others show where the mask misses deep water, and the `Deep Water` class handles those.

The plot shows the deep water index at every drop, by class. A good threshold sits above almost all shallow-class drops. This is how Lee et al. (2022) set it, from labelled points rather than automatically. Bear in mind that the `Deep Water` drops were placed where deep water is *hard* to mask, so many of them falling below the threshold is not a problem.

If a shallow class loses many drops, the threshold is too high. Lower it in notebook 1 and re-run from there.

In [ ]:
px, py = xr.DataArray(drops.geometry.x.values, dims="pt"), xr.DataArray(drops.geometry.y.values, dims="pt")
drops["shallow"] = comp["shallow"].sel(x=px, y=py, method="nearest").values == 1
hsv = rgb2hsv(np.clip(np.nan_to_num(np.dstack([comp["B1"], comp["B2"], comp["B3"]])), 0, 1))
dwi = comp["B3"].copy(data=(hsv[..., 1] - hsv[..., 0]) / (hsv[..., 1] + hsv[..., 0]))
drops["dwi"] = dwi.sel(x=px, y=py, method="nearest").values

fig, ax = plt.subplots(figsize=(10, 3.5))
for i, (cls, colour) in enumerate(zip(CLASSES, COLOURS)):
    v = drops.loc[drops["class"] == cls, "dwi"]
    ax.scatter(v, i + np.random.default_rng(i).uniform(-0.25, 0.25, len(v)), s=8, color=colour, edgecolor="k", lw=0.2)
ax.axvline(comp.attrs["deep_threshold"], color="r", label=f"DEEP_THRESHOLD in notebook 1 = {comp.attrs['deep_threshold']}")
ax.set(yticks=range(len(CLASSES)), yticklabels=CLASSES, xlabel="deep water index at the drop"); ax.legend(fontsize=8)
drops.groupby("class")["shallow"].mean().reindex(CLASSES).rename("share of drops kept (not deep, not land)").round(2)

## 3. Spectra by class

Before any modelling, look at what the satellite sees at each drop. Each drop's reflectance is the mean of a **3 × 3 pixel window** (30 m), which allows for GPS error (section 4). Classes whose spectra overlap will be confused by any classifier.

In [ ]:
window = comp[list(BANDS)].rolling(x=3, y=3, center=True, min_periods=1).mean()
spectra = window.sel(x=px, y=py, method="nearest").to_dataframe()[list(BANDS)]
spectra["class"] = drops["class"].values
spectra = spectra[drops["shallow"].values]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
wl = list(BANDS.values())
for cls, colour in zip(CLASSES, COLOURS):
    s = spectra.loc[spectra["class"] == cls, list(BANDS)]
    for ax, x in [(axes[0], wl), (axes[1], wl[:5])]:
        ax.plot(x, s.mean().values[:len(x)], "o-", color=colour, label=cls)
        ax.fill_between(x, (s.mean() - s.std()).values[:len(x)], (s.mean() + s.std()).values[:len(x)], color=colour, alpha=0.15)
axes[0].set(xlabel="wavelength (nm)", ylabel="reflectance", title="All bands: mean ± 1 SD")
axes[1].set(xlabel="wavelength (nm)", title="Visible and red edge only (B1–B5)"); axes[1].legend(fontsize=8)
plt.tight_layout()

Typically sand is clearly brighter than everything else, while seagrass, coral and algal mat overlap closely. Their differences are small, and partly a matter of depth rather than habitat. That is why the depth-invariant indices (section 7) and spatial validation (section 6) matter. NIR and SWIR carry almost no seabed signal underwater.

## 4. Position error in the ground truth

A drop's position is the **boat's** GPS position, with a few metres of error: typically ~6 m, occasionally more than 10 m. In these shallow (< 5 m), sheltered waters the camera lands almost under the boat, so GPS error is the main source of position error. In deeper water or a current, drift while the camera descends would add to it. Near a habitat edge, even a few metres can put the label on the wrong habitat. Two ways to allow for this:

1. **Sample a window, not a pixel.** Features are averaged over 3 × 3 pixels around each drop (used throughout).
2. **Flag drops in mixed surroundings.** Where the image varies a lot within GPS error of a drop, the recorded position may not be where the camera saw. We measure local variability (coefficient of variation of green in the same 3 × 3 window, ±15 m, about 2.5 times the GPS error), flag the most variable 15% as `edge`, and **leave them out of training**. They are kept for testing, because a map has to work at edges too.

In [ ]:
green = comp["B3"]
local_cv = green.rolling(x=3, y=3, center=True, min_periods=1).std() / green.rolling(x=3, y=3, center=True, min_periods=1).mean()
drops["local_cv"] = local_cv.sel(x=px, y=py, method="nearest").values
edge_limit = drops.loc[drops["shallow"], "local_cv"].quantile(0.85)
drops["edge"] = drops["local_cv"] > edge_limit

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(drops.loc[drops["shallow"], "local_cv"], bins=40, color="0.6")
ax.axvline(edge_limit, color="r", label="edge flag (top 15%)")
ax.set(xlabel="local variability of green within ±15 m", ylabel="drops"); ax.legend()
drops[drops["shallow"]].groupby("class")["edge"].mean().reindex(CLASSES).rename("share flagged as edge").round(2)

## 5. Spatial autocorrelation

Drops near each other usually see the same habitat: meadows, reef flats and sand patches are larger than the spacing between drops. And a survey day covers one area, so drops cluster. If neighbouring drops end up in training and test, the test is too easy.

The **correlogram** shows how often two drops share a class against the distance between them. Where it levels off is roughly how far apart training and test data need to be.

In [ ]:
use = drops[drops["shallow"]].reset_index(drop=True)
n_kept = use["class"].value_counts()
too_few = n_kept.index[n_kept < MIN_DROPS].tolist() + [c for c in CLASSES if c not in n_kept.index]
if too_few:
    print(f"left out (fewer than {MIN_DROPS} drops outside the deep water mask): {too_few}")
    use = use[~use["class"].isin(too_few)].reset_index(drop=True)
    CLASSES, COLOURS = [c for c in CLASSES if c not in too_few], [k for c, k in zip(CLASSES, COLOURS) if c not in too_few]
xy, lab = np.c_[use.geometry.x, use.geometry.y], use["class"].to_numpy()
pairs = cKDTree(xy).query_pairs(r=10000, output_type="ndarray")
dist = np.linalg.norm(xy[pairs[:, 0]] - xy[pairs[:, 1]], axis=1)
agree = pd.Series(lab[pairs[:, 0]] == lab[pairs[:, 1]]).groupby(dist // 250 * 250 + 125).mean()
chance = (use["class"].value_counts(normalize=True) ** 2).sum()

ax = agree.plot(marker="o", ms=3, figsize=(7, 3.5), label="observed")
ax.axhline(chance, color="r", ls="--", label="chance")
ax.set(xlabel="distance between drops (m)", ylabel="P(same class)", title="Label correlogram"); ax.legend();

## 6. Spatially independent test data and folds

The area is divided into square **blocks** larger than the correlation range, and whole blocks are assigned:

* about 25% of blocks are held back as the **test set**, used once at the end. We pick a split where every class appears in both sets.
* the remaining **training** blocks are divided into 5 **folds** for spatial cross-validation (choosing features, and the area of applicability threshold)

In [ ]:
BLOCK_M = 1000        # from the correlogram
use["block"] = (use.geometry.x // BLOCK_M).astype(int).astype(str) + "_" + (use.geometry.y // BLOCK_M).astype(int).astype(str)

for seed in range(100):                     # first block split with every class in both sets
    train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(use, groups=use["block"]))
    counts = pd.crosstab(use["class"], np.isin(np.arange(len(use)), test_idx))
    if counts.shape == (len(CLASSES), 2) and (counts.values >= 3).all():
        break
else:
    raise ValueError("no block split has every class in both sets: try a smaller BLOCK_M")
use["split"] = "train"
use.loc[test_idx, "split"] = "test"

train_rows = use.index[use["split"] == "train"]
use["fold"] = -1
for k, (_, val) in enumerate(GroupKFold(5).split(np.zeros(len(train_rows)), groups=use.loc[train_rows, "block"])):
    use.loc[train_rows[val], "fold"] = k
print(f"split seed {seed}: {len(train_rows)} training drops, {len(test_idx)} test drops")
pd.crosstab(use["class"], use["split"]).reindex(CLASSES)

In [ ]:
def nearest_train_distance(is_test):
    return cKDTree(xy[~is_test]).query(xy[is_test])[0]

block_test = (use["split"] == "test").to_numpy()
random_test = np.random.default_rng(0).random(len(use)) < 0.25
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [1.4, 1]})
axes[0].imshow(rgb(view, 4), extent=[x0, x1, y0, y1])
axes[0].scatter(xy[:, 0], xy[:, 1], s=10, c=np.where(block_test, "red", "deepskyblue"), edgecolor="k", lw=0.3)
axes[0].set_title("Block split: training (blue), test (red)"); axes[0].axis("off")
for name, is_test in [("random 25% of drops", random_test), ("25% of blocks", block_test)]:
    axes[1].hist(nearest_train_distance(is_test), bins=np.logspace(1.5, 4, 30), histtype="step", lw=2, label=name)
axes[1].set(xscale="log", xlabel="test drop → nearest training drop (m)", ylabel="test drops"); axes[1].legend();

With a random split, many test drops have a training drop within ~100 m, often from the same survey day and habitat patch. The block split pushes them further apart, which is closer to mapping places nobody dropped a camera.

## 7. Water column correction: Lyzenga depth-invariant indices

Light is absorbed exponentially with depth, at a different rate in each band. For one bottom type (sand) at varying depth, the log reflectances of two bands $i$ and $j$ fall on a line whose slope is the ratio of their attenuation coefficients:

$$\frac{k_i}{k_j} = a + \sqrt{a^2 + 1}, \qquad a = \frac{\sigma_{ii} - \sigma_{jj}}{2\,\sigma_{ij}}$$

where $\sigma$ is the covariance of the log reflectances over the sand profile. Projecting each pixel onto the direction perpendicular to that line removes the depth effect:

$$DII_{ij} = \frac{\ln R_i - \frac{k_i}{k_j}\ln R_j}{\sqrt{(k_i/k_j)^2 + 1}}$$

We use every pair of B1–B4. The ratios are estimated **once, on Tongatapu**, and reused for Vava'u in notebook 3.

In [ ]:
profile = gpd.GeoSeries([MultiPolygon([Polygon(p) for p in [
    [(-175.0435979639603, -21.090320930665253), (-175.0429113184525, -21.089760360767716),
     (-175.04479959359898, -21.086076563108495), (-175.04544332376256, -21.086436938648976)],
    [(-175.0650430955991, -21.08872049573213), (-175.06418478871433, -21.088760536789497),
     (-175.06366980458347, -21.08527692442997), (-175.06452811146823, -21.08527692442997)],
    [(-175.0611377992734, -21.086798512288162), (-175.06092322255222, -21.087078803088886),
     (-175.0571895876035, -21.08571738567546), (-175.05748999501316, -21.08523688243382)],
    [(-175.1117285661099, -21.040180321086233), (-175.11211480420803, -21.042503442521436),
     (-175.11082734388089, -21.04242333548896)]]])], crs=4326).to_crs(CRS)

LYZ = ["B1", "B2", "B3", "B4"]
log_r = np.log(comp[LYZ].where(comp[LYZ] > 0))
on_profile = log_r.rio.write_crs(CRS).rio.clip(profile.values, crs=CRS, drop=True)
P = np.column_stack([on_profile[b].values.ravel() for b in LYZ])
P = P[np.isfinite(P).all(axis=1)]
cov = np.cov(P, rowvar=False)

k_ratio = {}
for i, j in combinations(range(4), 2):
    a = (cov[i, i] - cov[j, j]) / (2 * cov[i, j])
    k_ratio[(LYZ[i], LYZ[j])] = a + np.sqrt(a ** 2 + 1)
print(f"{len(P)} sand profile pixels")
pd.Series({f"k{bi[1]}/k{bj[1]}": round(v, 3) for (bi, bj), v in k_ratio.items()})

## 8. Three ways to describe each pixel

We compare three classifiers. All are Random Forests (50 trees), trained and tested on **the same drops**, so any difference comes from what they are given:

| Method | What the classifier sees | Idea |
|---|---|---|
| **Pixel RF** | the pixel: bands B1–B5, six depth-invariant indices, three ratios | the standard approach, as in GEE script 2 |
| **Object RF** | the same features, averaged over a *superpixel*: a small patch of similar, neighbouring pixels | two-stage: segment first, then classify. Used, at larger scale, by the Allen Coral Atlas |
| **Embedding RF** | 64 *embedding* bands from a pretrained foundation model (AlphaEarth Foundations) | the model has learned a general description of each 10 m pixel from many satellites; we only train the last step |

### 8a. Spectral features (Pixel RF)

Bands B1–B5, the six depth-invariant indices, and three normalised differences: NDVI (B8A, B4), red-edge NDVI (B5, B4) and red–green (B4, B3). Features at each drop are the mean of the 3 × 3 window.

In [ ]:
def norm_diff(a, b):
    return (a - b) / (a + b)

feats = comp[["B1", "B2", "B3", "B4", "B5"]].copy()
for (bi, bj), k in k_ratio.items():
    feats[f"DII{bi[1]}{bj[1]}"] = (log_r[bi] - k * log_r[bj]) / np.sqrt(k ** 2 + 1)
feats["NDVI"] = norm_diff(comp["B8A"], comp["B4"])
feats["ReNDVI"] = norm_diff(comp["B5"], comp["B4"])
feats["RedGreen"] = norm_diff(comp["B4"], comp["B3"])
FEATURES = list(feats.data_vars)

table = feats[FEATURES].to_dataframe()[FEATURES].astype("float32")    # one row per pixel, in (y, x) order
ok = (comp["shallow"].values.ravel() == 1) & np.isfinite(table.to_numpy()).all(axis=1)

ux, uy = xr.DataArray(xy[:, 0], dims="pt"), xr.DataArray(xy[:, 1], dims="pt")
X_pixel = feats.rolling(x=3, y=3, center=True, min_periods=1).mean().sel(x=ux, y=uy, method="nearest").to_dataframe()[FEATURES].reset_index(drop=True)
rows = comp.indexes["y"].get_indexer(xy[:, 1], method="nearest")
cols = comp.indexes["x"].get_indexer(xy[:, 0], method="nearest")
drop_pixel = rows * comp.sizes["x"] + cols                             # each drop's row in `table`
print(f"{ok.sum():,} shallow pixels to classify")

### 8b. Superpixels (Object RF)

**SLIC** groups neighbouring pixels with similar colour (B1–B5) into superpixels of roughly `SEGMENT_PX` pixels. Each superpixel gets the mean of its pixels' features, and is classified as a whole. Land and deep water are set to one flat value so superpixels do not straddle them.

Why it may help: averaging removes pixel noise, a map of patches looks more like a habitat map, and a superpixel (here ~0.3 ha, about 55 m across) is larger than the GPS error. Why it may hurt: small or narrow habitat patches merge into their neighbours. `COMPACTNESS` trades regular shapes (high) against following the image (low).

In [ ]:
SEGMENT_PX = 30        # average superpixel size in pixels (30 × 100 m² = 0.3 ha)
COMPACTNESS = 0.5
SEG_BANDS = ["B1", "B2", "B3", "B4", "B5"]

ny, nx = comp.sizes["y"], comp.sizes["x"]
shallow_grid = ok.reshape(ny, nx)
img = np.dstack([comp[b].values for b in SEG_BANDS]).astype("float32")
img = (img - img[shallow_grid].mean(axis=0)) / img[shallow_grid].std(axis=0)
img[~shallow_grid] = -10                                  # land and deep water: one flat value, discarded below
segments = slic(img, n_segments=ny * nx // SEGMENT_PX, compactness=COMPACTNESS, channel_axis=-1, start_label=1, max_num_iter=5)
seg = segments.ravel()
seg_table = table[ok].groupby(seg[ok]).mean()             # mean features of the shallow pixels in each superpixel
X_object = seg_table.reindex(seg[drop_pixel]).reset_index(drop=True)
print(f"{len(seg_table):,} superpixels over shallow water; median size {pd.Series(seg[ok]).value_counts().median():.0f} pixels")

cx, cy = np.median(xy, axis=0)                            # zoom window used for the maps: move it to look elsewhere
zoom = dict(x=slice(cx - 2000, cx + 2000), y=slice(cy + 2000, cy - 2000))
z = comp.sel(**zoom)
zx0, zx1, zy0, zy1 = float(z.x[0]), float(z.x[-1]), float(z.y[-1]), float(z.y[0])
in_zoom = (xy[:, 0] > zx0) & (xy[:, 0] < zx1) & (xy[:, 1] > zy0) & (xy[:, 1] < zy1)
edges = find_boundaries(xr.DataArray(segments, coords={"y": comp.y, "x": comp.x}).sel(**zoom).values)
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(rgb(z, 4), extent=[zx0, zx1, zy0, zy1])
ax.imshow(np.where(edges, 1.0, np.nan), cmap=ListedColormap(["yellow"]), extent=[zx0, zx1, zy0, zy1], alpha=0.6, interpolation="nearest")
ax.scatter(xy[in_zoom, 0], xy[in_zoom, 1], s=18, c=[COLOURS[CLASSES.index(c)] if c in CLASSES else "w" for c in lab[in_zoom]], edgecolor="k", lw=0.4)
ax.set_title("Superpixel boundaries (yellow) and drops, 4 × 4 km"); ax.axis("off");

### 8c. Foundation model embeddings (Embedding RF)

**AlphaEarth Foundations** (Google DeepMind, 2025) summarises a year of observations from many sensors (Sentinel-2, Landsat, Sentinel-1 radar and others) into **64 numbers per 10 m pixel**: an *embedding*. It was trained without labels, to describe the Earth's surface in general, and is meant to be used exactly like this, with a few labelled points and a simple classifier.

Things to keep in mind:
* it is trained mostly for land; how well it describes the seabed is an open question, which is what this comparison tests
* it has no explicit water column correction (no Lyzenga step), and the embeddings are **annual** (we use 2023, the year of the composite)
* the dimensions have no physical meaning, so the spectra-style checks of section 3 are not possible

The embeddings are read for the AOI from the free copy on **Source Cooperative** (about 0.5 GB). The index of tiles is downloaded once (cached in `data/`). Each tile is resampled onto the composite's grid using its own georeferencing, so tiles in other UTM zones line up correctly.

*The AlphaEarth Foundations Satellite Embedding dataset is produced by Google and Google DeepMind* (CC-BY 4.0).

In [ ]:
AEF_ROOT = "https://data.source.coop/tge-labs/aef/v1/annual"
AEF_YEAR = 2023
AOI = (-175.36, -21.20, -174.99, -20.99)        # as notebook 1

if not (DATA / "aef_index.parquet").exists():
    urllib.request.urlretrieve(f"{AEF_ROOT}/aef_index.parquet", DATA / "aef_index.parquet")
aef_index = gpd.read_parquet(DATA / "aef_index.parquet")

def load_embeddings(aoi, like):
    """AlphaEarth embeddings (64 bands, int8, -128 = no data) on the grid of `like`."""
    tiles = aef_index[(aef_index["year"] == AEF_YEAR) & aef_index.intersects(box(*aoi))]
    path = next(c for c in tiles.columns if tiles[c].astype(str).str.endswith(".tiff").all())   # column holding the file paths
    out = None
    for p in tiles[path]:
        with rioxarray.open_rasterio(f"{AEF_ROOT}/{p.split('annual/')[-1]}", chunks=True) as src:
            part = src.rio.clip_box(*aoi, crs="EPSG:4326").load().rio.write_nodata(-128)
        part = part.rio.reproject_match(like, resampling=Resampling.nearest)
        out = part if out is None else out.where(out != -128, part)
    print(f"{len(tiles)} embedding tile(s) read")
    return out

def dequantise(q):
    q = q.astype("float32")
    return (q / 127.5) ** 2 * np.sign(q)

emb = load_embeddings(AOI, comp["B2"].rio.write_crs(CRS)).values      # 64 × y × x, int8
E = emb.reshape(64, -1)
has_emb = (E != -128).all(axis=0)
print(f"embeddings available for {has_emb[ok].mean():.0%} of shallow pixels")

EMB = [f"A{i:02d}" for i in range(64)]
def window_mean(r, c):                                    # 3 × 3 window, as for the spectral features
    w = emb[:, max(r - 1, 0):r + 2, max(c - 1, 0):c + 2].reshape(64, -1)
    w = w[:, (w != -128).all(axis=0)]
    return dequantise(w).mean(axis=1) if w.shape[1] else np.full(64, np.nan)
X_embed = pd.DataFrame([window_mean(r, c) for r, c in zip(rows, cols)], columns=EMB)

false_colour = np.dstack([dequantise(emb[i]) for i in (1, 16, 9)])
lo, hi = np.percentile(false_colour[has_emb.reshape(ny, nx)], [2, 98], axis=0)     # stretch each band
false_colour = np.clip((false_colour - lo) / (hi - lo), 0, 1)
fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(np.where(has_emb.reshape(ny, nx)[..., None], false_colour, 1))
ax.set_title("Three of the 64 embedding bands as RGB (white: no embedding)"); ax.axis("off");

## 9. The same training and test drops for all three

A drop is used only if all three methods have features for it. Training drops in the same superpixel as a test drop are removed, so the Object RF cannot see a test drop's features during training.

In [ ]:
METHODS = {"Pixel RF": X_pixel, "Object RF": X_object, "Embedding RF": X_embed}
TAGS = {"Pixel RF": "pixel", "Object RF": "object", "Embedding RF": "embedding"}
complete = np.logical_and.reduce([np.isfinite(X.to_numpy()).all(axis=1) for X in METHODS.values()])
drop_seg = seg[drop_pixel]
split, edge = use["split"].to_numpy(), use["edge"].to_numpy()
is_fit = complete & (split == "train") & ~edge & ~np.isin(drop_seg, drop_seg[split == "test"])
is_test = complete & (split == "test")
y_fit, y_test, folds, edge_test = lab[is_fit], lab[is_test], use["fold"].to_numpy()[is_fit], edge[is_test]
print(f"training drops {is_fit.sum()} (edge drops left out), test drops {is_test.sum()}; "
      f"{(~complete).sum()} drops lack features for at least one method")

## 10. Train and test

For each method: **cross-validation on the training drops** with random folds (which mix neighbouring drops) and spatial folds (section 6), then the final model is trained on all training drops and assessed **once** on the held-out test blocks.

In [ ]:
models, preds, scores = {}, {}, {}
for name, X in METHODS.items():
    rf = RandomForestClassifier(n_estimators=N_TREES, random_state=0, n_jobs=-1)
    s = {}
    for cv_name, cv in [("CV random", KFold(5, shuffle=True, random_state=0)), ("CV spatial", PredefinedSplit(folds))]:
        p = cross_val_predict(rf, X[is_fit], y_fit, cv=cv)
        s[f"{cv_name}: accuracy"] = accuracy_score(y_fit, p)
        s[f"{cv_name}: balanced"] = balanced_accuracy_score(y_fit, p)
    models[name] = rf.fit(X[is_fit], y_fit)
    preds[name] = rf.predict(X[is_test])
    s["test: accuracy"] = accuracy_score(y_test, preds[name])
    s["test: balanced"] = balanced_accuracy_score(y_test, preds[name])
    s["test: kappa"] = cohen_kappa_score(y_test, preds[name])
    s["test: edge drops"] = accuracy_score(y_test[edge_test], preds[name][edge_test]) if edge_test.any() else np.nan
    s["test: other drops"] = accuracy_score(y_test[~edge_test], preds[name][~edge_test])
    scores[name] = s
pd.DataFrame(scores).round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
for ax, (name, p) in zip(axes, preds.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, p, labels=CLASSES, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name); ax.tick_params(axis="x", rotation=45)
    if ax is not axes[0]:
        ax.set(ylabel="", yticklabels=[])
plt.tight_layout()

f1 = pd.DataFrame({name: f1_score(y_test, p, labels=CLASSES, average=None, zero_division=0) for name, p in preds.items()}, index=CLASSES)
f1.round(2).rename_axis("F1 score, test drops")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, name in zip(axes, ["Pixel RF", "Object RF"]):
    imp = permutation_importance(models[name], METHODS[name][is_test], y_test, n_repeats=10, random_state=0)
    pd.Series(imp.importances_mean, FEATURES).sort_values().plot.barh(ax=ax, color="0.5")
    ax.set_title(f"{name}: permutation importance (test drops)")
plt.tight_layout()

## 11. Area of applicability

The test accuracy only holds for pixels that **resemble the training data**. The area of applicability (Meyer & Pebesma 2021) makes this explicit:

1. Standardise the features and weight each by its importance to the model
2. For every pixel, the distance to the *nearest training drop* in that feature space, divided by the mean distance between training drops: the **dissimilarity index (DI)**
3. A threshold from the spatial folds: how dissimilar each training drop is from the drops in *other* folds. The outlier-robust maximum of those values is the limit the cross-validation has tested.
4. Pixels above the threshold are **outside the area of applicability**: the model is extrapolating, and its accuracy there is unknown

Each method has its own feature space, so each gets its own area of applicability. Pixels outside are where a new drop-camera survey would add the most.

In [ ]:
def aoa_fit(model, X, y, folds):
    w = permutation_importance(model, X, y, n_repeats=5, random_state=0).importances_mean.clip(min=0)
    mu, sd = X.mean().to_numpy(), X.std().to_numpy()
    sd = np.where(sd > 0, sd, 1)
    keep = w > 0                                           # features the model does not use add nothing
    T = ((X.to_numpy() - mu) / sd * w)[:, keep]
    d_mean = pdist(T).mean()
    d_cv = np.empty(len(T))
    for k in np.unique(folds):
        d_cv[folds == k] = cKDTree(T[folds != k]).query(T[folds == k])[0]
    q1, q3 = np.percentile(d_cv / d_mean, [25, 75])
    return {"mu": mu, "sd": sd, "weights": w, "keep": keep, "train_scaled": T, "train_features": X,
            "d_mean": d_mean, "threshold": min(q3 + 1.5 * (q3 - q1), (d_cv / d_mean).max())}

def dissimilarity(parts, X):
    tree, X = cKDTree(parts["train_scaled"]), np.asarray(X, dtype="float32")
    di = [tree.query(((X[i:i + 200_000] - parts["mu"]) / parts["sd"] * parts["weights"])[:, parts["keep"]])[0]
          for i in range(0, len(X), 200_000)]
    return np.concatenate(di) / parts["d_mean"]

aoa_parts = {}
for name, X in METHODS.items():
    aoa_parts[name] = aoa_fit(models[name], X[is_fit], y_fit, folds)
    inside = dissimilarity(aoa_parts[name], X[is_test]) <= aoa_parts[name]["threshold"]
    scores[name]["test: inside AOA"] = accuracy_score(y_test[inside], preds[name][inside]) if inside.any() else np.nan
    scores[name]["test: outside AOA"] = accuracy_score(y_test[~inside], preds[name][~inside]) if (~inside).any() else np.nan
    scores[name]["test drops inside AOA"] = inside.mean()
pd.DataFrame(scores).loc[["test: inside AOA", "test: outside AOA", "test drops inside AOA"]].round(2)

## 12. Maps

Each model classifies every shallow pixel: Pixel RF pixel by pixel, Object RF superpixel by superpixel, Embedding RF wherever there are embeddings. The dissimilarity index is computed the same way, so each map comes with its own area of applicability. This takes a few minutes.

In [ ]:
def to_grid(values, where, fill, dtype):
    full = np.full(ok.size, fill, dtype=dtype)
    full[where] = values
    return xr.DataArray(full.reshape(ny, nx), coords={"y": comp.y, "x": comp.x})

code = {c: i for i, c in enumerate(CLASSES)}
maps, dis = {}, {}

maps["Pixel RF"] = to_grid(pd.Series(models["Pixel RF"].predict(table[ok])).map(code).values, ok, 255, "uint8")
dis["Pixel RF"] = to_grid(dissimilarity(aoa_parts["Pixel RF"], table[ok]), ok, np.nan, "float32")

seg_class = pd.Series(models["Object RF"].predict(seg_table), index=seg_table.index).map(code)
seg_di = pd.Series(dissimilarity(aoa_parts["Object RF"], seg_table), index=seg_table.index)
maps["Object RF"] = to_grid(seg_class.reindex(seg[ok]).values, ok, 255, "uint8")
dis["Object RF"] = to_grid(seg_di.reindex(seg[ok]).values, ok, np.nan, "float32")

ok_emb = ok & has_emb
E_ok = pd.DataFrame(dequantise(E[:, ok_emb].T), columns=EMB)
maps["Embedding RF"] = to_grid(pd.Series(models["Embedding RF"].predict(E_ok)).map(code).values, ok_emb, 255, "uint8")
dis["Embedding RF"] = to_grid(dissimilarity(aoa_parts["Embedding RF"], E_ok), ok_emb, np.nan, "float32")
del E_ok

for name in METHODS:
    scores[name]["mapped area inside AOA"] = float((dis[name] <= aoa_parts[name]["threshold"]).sum() / dis[name].notnull().sum())

kw = dict(cmap=ListedColormap(COLOURS), vmin=-0.5, vmax=len(CLASSES) - 0.5, interpolation="nearest")
legend = dict(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in COLOURS], labels=CLASSES, fontsize=8)
fig, axes = plt.subplots(3, 1, figsize=(14, 25))
for ax, name in zip(axes, METHODS):
    m, d = maps[name], dis[name]
    ax.imshow(rgb(comp, 4))
    ax.imshow(np.where(m == 255, np.nan, m), **kw)
    ax.imshow(np.where(d > aoa_parts[name]["threshold"], 1.0, np.nan), cmap=ListedColormap(["#ff00ff"]), alpha=0.5, interpolation="nearest")
    ax.set_title(f"{name}   (magenta wash: outside its area of applicability)"); ax.axis("off")
axes[0].legend(loc="lower left", **legend)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
axes[0].imshow(rgb(z, 4), extent=[zx0, zx1, zy0, zy1])
axes[0].scatter(xy[in_zoom, 0], xy[in_zoom, 1], s=18, c=[COLOURS[CLASSES.index(c)] if c in CLASSES else "w" for c in lab[in_zoom]], edgecolor="k", lw=0.4)
axes[0].set_title("Composite and drops")
for ax, name in zip(axes[1:], METHODS):
    m = maps[name].sel(**zoom)
    ax.imshow(rgb(z, 4), extent=[zx0, zx1, zy0, zy1])
    ax.imshow(np.where(m == 255, np.nan, m), extent=[zx0, zx1, zy0, zy1], **kw)
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
axes[-1].legend(loc="lower right", **legend)
plt.tight_layout()

Where do the three maps agree? Agreement is not accuracy, since all three can be wrong together, but disagreement marks pixels whose class depends on the method: those are the least certain parts of any one map.

In [ ]:
stack = np.stack([maps[name].values for name in METHODS])
valid = (stack != 255).all(axis=0)
n_classes = 1 + (stack[0] != stack[1]) + ((stack[2] != stack[0]) & (stack[2] != stack[1]))   # distinct classes among the three
agreement = np.where(valid, n_classes, 0)

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(rgb(comp, 4))
ax.imshow(np.where(valid, agreement, np.nan), cmap=ListedColormap(["#1a9850", "#fee08b", "#d73027"]), vmin=0.5, vmax=3.5, interpolation="nearest")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#1a9850", "#fee08b", "#d73027"]],
          labels=["all three agree", "two agree", "all differ"], loc="lower left")
ax.set_title("Agreement between the three maps"); ax.axis("off")

pixel_ha = abs(float(comp.x[1] - comp.x[0])) ** 2 / 1e4
print(f"all three agree on {(agreement[valid] == 1).mean():.0%} of pixels; pairwise agreement:")
for a, b in combinations(METHODS, 2):
    print(f"   {a} vs {b}: {(maps[a].values == maps[b].values)[valid].mean():.0%}")
pd.DataFrame({name: pd.Series(maps[name].values[valid]).map(dict(enumerate(CLASSES))).value_counts().reindex(CLASSES, fill_value=0) * pixel_ha
              for name in METHODS}).round(0).rename_axis("area (ha)")

## 13. Comparison

All the numbers in one table. When comparing, ask:
* Does a method's advantage hold up under **spatial** cross-validation and on the **test blocks**, or only with random folds?
* Which classes gain or lose (the F1 table in section 10), in particular sand and algal mat?
* How much of the map lies inside each method's area of applicability, and how accurate is each method there?

In [ ]:
pd.DataFrame(scores).round(2)

## 14. Save

The three maps, and one file with the three models and everything notebook 3 needs: feature lists, Lyzenga ratios, superpixel settings, embedding source and the area of applicability for each.

In [ ]:
for name, tag in TAGS.items():
    path = DATA / f"tongatapu_classes_{tag}.tif"
    path.unlink(missing_ok=True)
    maps[name].rio.write_crs(CRS).rio.write_nodata(255).rio.to_raster(path)
joblib.dump({"models": models, "aoa": aoa_parts, "features": FEATURES, "classes": CLASSES, "colours": COLOURS, "k_ratio": k_ratio,
             "segmentation": {"bands": SEG_BANDS, "segment_px": SEGMENT_PX, "compactness": COMPACTNESS},
             "aef": {"root": AEF_ROOT, "year": AEF_YEAR}}, DATA / "rf_models.joblib")
print("saved data/tongatapu_classes_{pixel,object,embedding}.tif and data/rf_models.joblib")

## Exercises
1. **Superpixel size.** Try `SEGMENT_PX` = 10 and 100. At what size does the Object RF stop helping, and which classes suffer first?
2. **Edge drops.** Superpixels are larger than the GPS error. Train with the edge drops included (section 9): does the Object RF cope better than the Pixel RF?
3. **Combine.** Concatenate the spectral features and the embeddings into one table and train a fourth model. Does it beat both?
4. **Block size.** Try `BLOCK_M` = 300 and 3000. Does the ranking of the three methods change?
5. **Next survey.** Where would you send the boat to expand the area of applicability of the best method?

**References:** Lyzenga (1978) *Applied Optics* 17: 379 · Achanta et al. (2012) SLIC superpixels, *IEEE TPAMI* 34: 2274 · Lyons et al. (2020) *Remote Sens. Ecol. Conserv.* 6: 557 · Brown et al. (2025) AlphaEarth Foundations, arXiv:2507.22291 · Meyer & Pebesma (2021) *Methods Ecol. Evol.* 12: 1620